# functional-module-wrap — ex2: MyLeakyReLU — parametric F.leaky_relu wrap with extra_repr

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `functional-module-wrap`. Running the final beacon cell reports progress against the `PyTorch: functional module wrap` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: functional module wrap` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`functional-module-wrap`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "functional-module-wrap"
DD_SUBTOPIC = "PyTorch: functional module wrap"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Parametric functional wrap — `F.leaky_relu(negative_slope, inplace)`

Ex1 wrapped the parameter-free `F.relu`. The deepening move handles a PARAMETRIC functional — `F.leaky_relu(x, negative_slope=0.01, inplace=False)`. The Module must STORE the parameters (not the tensor weights — these aren't learnable, they're hparams) and pass them through:

```python
class MyLeakyReLU(nn.Module):
    def __init__(self, negative_slope=0.01, inplace=False):
        super().__init__()
        self.negative_slope = negative_slope
        self.inplace = inplace

    def forward(self, x):
        return F.leaky_relu(x, self.negative_slope, self.inplace)

    def extra_repr(self):
        return f'negative_slope={self.negative_slope}, inplace={self.inplace}'
```

**`extra_repr` over `__repr__`.** `nn.Module.__repr__` already handles the class name + children; `extra_repr` slots your hparams into that format. This is how `nn.LeakyReLU(negative_slope=0.2)` shows up in `print(model)`.

**Still no parameters.** Hparams are stored as plain Python attrs, not `nn.Parameter`. `model.parameters()` stays empty — these are config.

### Exercise 2 — MyLeakyReLU — parametric F.leaky_relu wrap with extra_repr

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the parametric functional-to-Module wrap pattern: store `negative_slope` and `inplace` as plain attributes, delegate `forward` to `F.leaky_relu(x, negative_slope, inplace)`, and expose the hparams via `extra_repr`.
> Keywords: leaky-relu, module, extra_repr, parametric
> ```

**KCs targeted:** `parametric-functional-wrap`, `extra_repr-hparam-display`

Implement `MyLeakyReLU(nn.Module)`. Like ex1's MyReLU but PARAMETRIC.

Constraints:
1. Subclass `nn.Module`. Call `super().__init__()` in `__init__`.
2. `__init__(self, negative_slope=0.01, inplace=False)` stores BOTH parameters as plain Python attributes (NOT as `nn.Parameter` — these are config, not learnables).
3. `forward(x)` returns `F.leaky_relu(x, self.negative_slope, self.inplace)`. Do NOT delegate to `nn.LeakyReLU` internally — exercise the functional wrap directly.
4. `extra_repr(self)` returns `f'negative_slope={self.negative_slope}, inplace={self.inplace}'` so `print(model)` shows the hparams.

Output: an `nn.Module` subclass that matches `nn.LeakyReLU` numerically and has zero `model.parameters()` entries.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class MyLeakyReLU(nn.Module):
    def __init__(self, negative_slope=0.01, inplace=False):
        super().__init__()
        self.negative_slope = negative_slope
        self.inplace = inplace

    def forward(self, x):
        return F.leaky_relu(x, self.negative_slope, self.inplace)

    def extra_repr(self):
        return f'negative_slope={self.negative_slope}, inplace={self.inplace}'


<details><summary>Solution</summary>

```python
import torch.nn as nn
import torch.nn.functional as F

class MyLeakyReLU(nn.Module):
    def __init__(self, negative_slope=0.01, inplace=False):
        super().__init__()
        self.negative_slope = negative_slope
        self.inplace = inplace

    def forward(self, x):
        return F.leaky_relu(x, self.negative_slope, self.inplace)

    def extra_repr(self):
        return f'negative_slope={self.negative_slope}, inplace={self.inplace}'
```

**Hparams as plain attrs, not Parameters.** `nn.Parameter` would make `negative_slope` show up in `model.parameters()` and downstream optimizer construction would try to update it. The intent is config: the slope is fixed at construction time.

**`extra_repr` returns a single line.** `nn.Module.__repr__` wraps it in the `ClassName(...)` envelope and handles indentation for nested modules. You just supply the comma-joined hparam string.

**`inplace=True` is a performance flag.** It writes the result into the input's storage instead of allocating a new tensor. Saves memory in long chains of activations but breaks autograd if the input is needed for the backward pass — that's why the default is `False`.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()